# 28 - Offline Political-Message Pool Smoke Test

Mini validation for the new offline political-broadcast path shipped in v0.6.

**What this notebook checks**

1. The pre-authored message set (`v1`) loads cleanly via `load_message_pool`.
2. Every `(side, policy)` cell expected by a single-policy run is non-empty.
3. A 5-agent / 2-day / single-policy run with `political_message_source="offline"` (the new default) executes end-to-end without any LLM call for the political broadcast.
4. Every `political_broadcast` row in `results['messages']` carries a non-empty `political_message_id`, and those ids resolve back to the shipped CSV.
5. (Optional, gated) The same config re-run with `political_message_source="llm"` produces empty `political_message_id` values - parity check that the LLM path is untouched.

**Runtime defaults to the local Qwen3 server** (`mlx_lm.server` on `http://localhost:8080/v1`), matching `notebooks/25_local_llm_integrated_smoke.ipynb`. No API credentials required.

This is a **precursor** to the full NB-style canonical smoke run. Keep it small and fast.


## 1. Imports + standalone pool load

First, load the pool without touching the simulator. Confirms the shipped CSVs are valid and shows per-cell sizes.

In [1]:
import os, sys, random, logging

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('httpcore').setLevel(logging.WARNING)

import pandas as pd

from cag.abm.political_messages import load_message_pool, PACKAGE_KEY
from cag.abm.attributes.opinion import ALL_CLIMATE_POLICIES, ClimatePolicyID

pool = load_message_pool('v1', seed=42)

rows = []
for side in ('A', 'B'):
    for pid in ALL_CLIMATE_POLICIES:
        rows.append({'side': side, 'policy_id': str(pid), 'cell_size': pool.cell_size(side, pid)})
    rows.append({'side': side, 'policy_id': PACKAGE_KEY, 'cell_size': pool.cell_size(side, PACKAGE_KEY)})

cell_df = pd.DataFrame(rows)
print(f'Total cells: {len(cell_df)} (12 single-policy + 2 package)')
print(f'Empty cells: {(cell_df.cell_size == 0).sum()} (must be 0)')
cell_df

INFO Loaded offline message pool 'v1': 14 cells, 242 messages, 24 sources (seed=42)


Total cells: 14 (12 single-policy + 2 package)
Empty cells: 0 (must be 0)


,side,policy_id,cell_size
0,A,ClimatePolicyID(1),20
1,A,ClimatePolicyID(2),20
2,A,ClimatePolicyID(3),20
3,A,ClimatePolicyID(4),20
4,A,ClimatePolicyID(5),20
5,A,ClimatePolicyID(6),20
6,A,PACKAGE,1
7,B,ClimatePolicyID(1),20
8,B,ClimatePolicyID(2),20
9,B,ClimatePolicyID(3),20


## 2. Build a tiny SurveyedNation (5 agents)

Same construction pattern as NB16/NB17, scaled down to 5 agents.

In [2]:
from cag.io.survey import load
from cag.abm.agent import SurveyedCitizen, PoliticalAgent
from cag.abm.environment import SurveyedNation

from gabm.abm.attributes.gender import GenderMap, GenderID
from gabm.abm.attributes.politics import PoliticsID
from gabm.abm.democracy.election import ElectionID
from cag.abm.attributes.region import UKRegionMap, RegionID
from cag.abm.attributes.education import SurveyEducationMap, EducationID
from cag.abm.attributes.ethnicity import SurveyEthnicityMap, EthnicityID
from cag.abm.attributes.income import SurveyIncomeMap, IncomeID
from cag.abm.attributes.politics import SurveyPoliticsMap
from cag.abm.attributes.family import SurveyFamilyMap, FamilyID
from cag.abm.democracy.elections.ukge2019 import UKGE2019VoteMap, UKGE2019VoteID
from cag.abm.democracy.elections.brexit import BrexitVoteMap, BrexitVoteID
from cag.abm.attributes.narratives import (
    SelftranscMap, SelfenhMap, OpennessMap, ConformTradMap, SDOMap, EDOMap, RWAMap,
    rescale_1_6, rescale_1_7,
)

N_CITIZENS = 5
RANDOM_SEED = 42
year = 2026
random.seed(RANDOM_SEED)

UKGE2019_ELECTION_ID = ElectionID(0)
BREXIT_REFERENDUM_ID = ElectionID(1)

sn = SurveyedNation(
    year=year, place='UK',
    gender_map=GenderMap(),
    region_map=UKRegionMap(),
    education_map=SurveyEducationMap(),
    ethnicity_map=SurveyEthnicityMap(),
    income_map=SurveyIncomeMap(),
    politics_map=SurveyPoliticsMap(),
    family_map=SurveyFamilyMap(),
    ukge2019_vote_map=UKGE2019VoteMap(UKGE2019_ELECTION_ID),
    brexit_vote_map=BrexitVoteMap(BREXIT_REFERENDUM_ID),
    selftransc_map=SelftranscMap,
    selfenh_map=SelfenhMap,
    openness_map=OpennessMap,
    conformtrad_map=ConformTradMap,
    sdo_map=SDOMap,
    edo_map=EDOMap,
    rwa_map=RWAMap,
)

data = load('../data/yougov_survey_data/YouGovProcessedData.csv')
data = data.sample(n=N_CITIZENS, random_state=RANDOM_SEED).reset_index(drop=True)

for i in range(len(data)):
    row = data.iloc[i]
    sc = SurveyedCitizen(
        agent_id=row.get('ID', None), environment=sn,
        year_of_birth=year - int(row.get('age', 0)),
        gender_id=GenderID.MALE if int(row.get('male_dummy', 0)) == 1 else GenderID.FEMALE,
        region_id=RegionID(int(row.get('tprofile_GOR', 0))),
        education_id=EducationID(int(row.get('profile_education_level', 0))),
        income_id=IncomeID(int(row.get('tprofile_gross_household', 0))),
        ethnicity_id=EthnicityID(int(row.get('ethnicity_R', 0))),
        family_id=FamilyID.PARENT if int(row.get('parent_dummy', 0)) == 1 else FamilyID.NOT_PARENT,
        ukge2019_vote_id=UKGE2019VoteID(int(row.get('Vote2019R', 0))),
        brexit_vote_id=BrexitVoteID(int(row.get('pastvote_EURef', 0))),
        politics_id=PoliticsID(int(row.get('Political_Left_Right', 0))),
        selftransc_id=rescale_1_6(int(row.get('Selftransc_Val', 0))),
        selfenh_id=rescale_1_6(int(row.get('Selfenh_Values', 0))),
        openness_id=rescale_1_6(int(row.get('Openness', 0))),
        conformtrad_id=rescale_1_6(int(row.get('ConformTrad', 0))),
        sdo_id=rescale_1_7(int(row.get('SDO', 0))),
        edo_id=rescale_1_7(int(row.get('EDO', 0))),
        rwa_id=rescale_1_6(int(row.get('RWA', 0))),
        original_survey_data=data.iloc[i],
    )
    sn.agents_active[sc.id] = sc

print(f'Citizens loaded: {len(sn.agents_active)}')

INFO Columns with NaN counts (before filtering):
tprofile_gross_household    398
Political_Left_Right          6
dtype: int64
INFO Column with most NaNs: tprofile_gross_household (398 NaNs)
INFO 1483 rows after filtering.


Citizens loaded: 5


## 3. Run offline single-policy simulation (default)

2 days, two distinct policies, default `political_message_source="offline"`. The political broadcast text comes entirely from the v1 message set; no LLM call is made for `agent.generate_message`. Citizen reflections + end-of-day surveys still use the LLM — here served locally via `mlx_lm.server` (Qwen3-8B-4bit), matching the canonical local-LLM smoke pattern from `25_local_llm_integrated_smoke.ipynb`. No API credentials required.

**Prereq:** `mlx_lm.server` must already be running on `http://localhost:8080/v1`. The simulator's `_resolve_runtime` calls `ping_local()` at start and will fail fast if the server is unreachable.


In [3]:
from cag.abm.sim import run_simulation

N_DAYS = 2

# Local LLM defaults (NB 25 pattern). Override these two only if your server
# is on a different host/port or you've loaded a different model.
LOCAL_BASE_URL = "http://localhost:8080/v1"
LOCAL_MODEL = "mlx-community/Qwen3-8B-4bit"

config = {
    'n_citizens': N_CITIZENS,
    'communication_mode': 'single_policy',
    'days': [
        {'policy': ClimatePolicyID.CARBON_TAX, 'phases': ['P-A', 'P-B', 'C']},
        {'policy': ClimatePolicyID.RENEWABLE_ENERGY, 'phases': ['P-A', 'P-B', 'C']},
    ],
    'k_peers_per_day': 2,
    # ---- Local LLM via the integrated provider ----
    'llm_provider': 'local',
    'llm_model': LOCAL_MODEL,
    'local_base_url': LOCAL_BASE_URL,
    # ------------------------------------------------
    'llm_temperature': 0.5,
    'thinking': False,
    'debias': False,
    'p_intra': 0.15,
    'p_inter': 0.02,
    'random_seed': RANDOM_SEED,
    # New keys exercised by this notebook:
    'political_message_source': 'offline',
    'political_message_set': 'v1',
}

# Wrap the political agents' generate_message so we can prove the LLM is NEVER
# invoked for the broadcast text in offline mode. This is the key invariant.
from cag.abm.agent import PoliticalAgent
_orig_generate_message = PoliticalAgent.generate_message
_llm_call_count = {'n': 0}
def _trip(self, *args, **kwargs):
    _llm_call_count['n'] += 1
    return _orig_generate_message(self, *args, **kwargs)
PoliticalAgent.generate_message = _trip

try:
    results_offline = run_simulation(config, sn)
finally:
    PoliticalAgent.generate_message = _orig_generate_message

df_messages = results_offline['messages']
print(f'messages rows: {len(df_messages)}')
print(f'PoliticalAgent.generate_message invocations: {_llm_call_count["n"]} (must be 0)')


INFO Loaded offline message pool 'v1': 14 cells, 242 messages, 24 sources (seed=42)
INFO Reach subsample: agent_a audience 3/3 (reach_a=1.0)
INFO Reach subsample: agent_b audience 3/3 (reach_b=1.0)
INFO Simulation: 5 agents, 2 days
INFO Running baseline survey (day 0), policy=ClimatePolicyID(5), anchor=llm_survey
INFO --- Day 1/2 (policy=ClimatePolicyID(5), phases=['P-A', 'P-B', 'C']) ---
INFO [P-A] Day 1: delivered message to 3 citizens. Message (first 120 chars): Households need protection from rising energy costs during the transition. That protection is best delivered through inv...
INFO [P-A] Sample reflection: This idea of using a carbon tax to fund insulation and public transport resonates with me because it addresses both environmental and social concerns. I’ve always believed in equal opportunities, and ...
INFO [P-B] Day 1: delivered message to 3 citizens. Message (first 120 chars): Scrap the carbon tax on generation, slash electricity bills by around twenty percent and let i

messages rows: 16
PoliticalAgent.generate_message invocations: 0 (must be 0)


## 4. Verify message_log ids resolve back to the v1 CSV

Every `political_broadcast` row must have a non-empty `political_message_id` matching a real id in `messages_v1.csv`, and the `message_text` column must match the canonical text for that id.

In [4]:
broadcasts = df_messages[df_messages['message_type'] == 'political_broadcast'].copy()
print(f'political_broadcast rows: {len(broadcasts)}')

assert (broadcasts['political_message_id'] != '').all(), \
    'offline mode should populate political_message_id on every broadcast row'

# Pool sides are 'A'/'B'; message_log sender_side is 'pro_climate'/'anti_climate'.
SIDE_LETTER = {'pro_climate': 'A', 'anti_climate': 'B'}

# Look every id up via the pool and confirm text + side + policy match.
mismatches = []
for _, row in broadcasts.iterrows():
    rec = pool.get_record(row['political_message_id'])
    if rec is None:
        mismatches.append((row['political_message_id'], 'unknown id'))
        continue
    if rec.message_text != row['message_text']:
        mismatches.append((row['political_message_id'], 'text mismatch'))
    expected_letter = SIDE_LETTER.get(row['sender_side'])
    if rec.side != expected_letter:
        mismatches.append((row['political_message_id'], f'side mismatch {rec.side}!={expected_letter} (sender_side={row["sender_side"]})'))

print(f'mismatches: {len(mismatches)} (must be 0)')
if mismatches:
    print(mismatches[:5])

print('\nUnique ids used:')
print(broadcasts['political_message_id'].value_counts())


political_broadcast rows: 12
mismatches: 0 (must be 0)

Unique ids used:
political_message_id
A_05_12    3
B_05_18    3
A_01_17    3
B_01_07    3
Name: count, dtype: int64


## 5. Optional - LLM parity re-run

Skip unless you want to spend wall-time generating broadcasts on the local server. Re-runs the same config with `political_message_source="llm"` and asserts that every broadcast row now has an **empty** `political_message_id` (LLM path untouched, no pool reference). Uses the same local Qwen3 provider — no API credentials.


In [ ]:
RUN_LLM_PARITY = False  # flip to True to execute

if RUN_LLM_PARITY:
    # Rebuild a fresh nation - re-run sections 2 above first, or factor out into a helper.
    # Here we just mutate the config; the user is responsible for the nation state.
    cfg_llm = dict(config)
    cfg_llm['political_message_source'] = 'llm'
    results_llm = run_simulation(cfg_llm, sn)
    df_llm = results_llm['messages']
    llm_broadcasts = df_llm[df_llm['message_type'] == 'political_broadcast']
    assert (llm_broadcasts['political_message_id'] == '').all(), \
        'llm mode should leave political_message_id empty on every broadcast row'
    print(f'PASS: {len(llm_broadcasts)} llm-mode broadcasts, all with empty political_message_id')
else:
    print('Skipped (set RUN_LLM_PARITY=True to execute).')